# Appendix B — Python Cheat-Sheet for Economists

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/appB_python_cheatsheet.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## B.1 Python basics → Ch 2, 3

In [ ]:
import math

g = 0.07                                   # a variable (7 per cent as a decimal)
years = math.log(2) / math.log(1 + g)      # exact doubling time
print(f"Doubling time at {g:.0%}: {years:.1f} years")   # f-string formatting

slabs = [(400_000, 0.00), (800_000, 0.05)]  # list of tuples; _ separates digits
rates = {"repo": 5.5, "crr": 4.0}           # dictionary: name -> value
for limit, rate in slabs:                   # loop with tuple unpacking
    print(limit, rate)
squares = [x ** 2 for x in range(5)]        # list comprehension


def cagr(first, last, n):
    """Compound annual growth rate between two levels n years apart."""
    return (last / first) ** (1 / n) - 1


print(squares, f"{cagr(100, 200, 10):.2%}", "repo" in rates)

## B.2 Getting data → Ch 4

In [ ]:
import numpy as np
import pandas as pd

from indiadata import bis, mospi_cpi, mospi_nas

cpi = mospi_cpi()                          # read from data/raw (cached)
repo = bis("WS_CBPOL", "M.IN")
pc = mospi_nas(25)                         # per capita NSDP by state, rupees
print(cpi.shape, repo.shape, pc.shape)

## B.3 pandas: look, select, create → Ch 6

In [ ]:
# cpi.info() lists columns, types and missing values; describe() summarises
cpi[["index", "inflation"]].describe().round(2)

In [ ]:
recent = cpi.loc[cpi["date"] >= "2024-01-01", ["date", "index", "inflation"]]
high = cpi[cpi["inflation"].between(6, 8)]             # a range
big = pc[pc["state"].isin(["Bihar", "Kerala"]) & (pc["fy"] == 2023)]
cpi = cpi.assign(above_band=cpi["inflation"] > 6,      # new columns
                 regime=np.where(cpi["date"] >= "2016-10-01", "FIT", "before"))
print(len(recent), len(high))
big[["state", "year", "current_price"]]

## B.4 pandas: change over time → Ch 1, 6, 10, 11

In [ ]:
s = cpi.set_index("date")["index"]
yoy = s.pct_change(12) * 100               # year-on-year % (monthly data)
mom = s.pct_change() * 100                 # month-on-month %
lag1 = s.shift(1)                          # previous month
d1 = s.diff()                              # first difference
vol = yoy.rolling(24).std()                # 24-month rolling SD
q = yoy.resample("QE").mean()              # monthly -> quarterly average
pd.DataFrame({"yoy": yoy, "mom": mom, "vol": vol}).dropna().tail(3).round(2)

In [ ]:
pc = pc.sort_values(["state", "fy"])
pc["growth"] = pc.groupby("state")["constant_price"].pct_change() * 100  # within state
pc["lag"] = pc.groupby("state")["constant_price"].shift(1)
pc[pc["state"] == "Bihar"][["year", "constant_price", "growth"]].tail(3).round(1)

In [ ]:
start = s.index.year.where(s.index.month >= 4, s.index.year - 1)
fy = pd.Series(start.astype(str) + "-" + (start + 1).astype(str).str[-2:],
               index=s.index)
fy.loc["2024-02-01":"2024-05-01"]

## B.5 pandas: group, reshape, merge → Ch 6, 7

In [ ]:
by_year = (cpi.assign(year=cpi["date"].dt.year)
              .groupby("year")["inflation"].agg(["mean", "max", "count"]))
cpi["dev"] = cpi["inflation"] - cpi.groupby(cpi["date"].dt.year)["inflation"].transform("mean")
by_year.tail(3).round(2)

In [ ]:
wide = pc.pivot(index="state", columns="fy", values="current_price")   # long -> wide
long = wide.reset_index().melt(id_vars="state", var_name="fy",
                               value_name="current_price")              # wide -> long
print(wide.shape, long.shape)

In [ ]:
m = cpi.merge(repo.rename(columns={"value": "repo"}), on="date",
              how="left", validate="one_to_one", indicator=True)
print(m["_merge"].value_counts().to_dict())
assert not m.duplicated("date").any()      # keys are unique

## B.6 Weighted statistics for survey data → Ch 14, 15

In [ ]:
x = np.array([10_000, 15_000, 40_000])     # e.g. monthly earnings
w = np.array([300, 100, 50])               # survey weights
mean_w = np.average(x, weights=w)          # weighted mean
order = np.argsort(x)
cum = np.cumsum(w[order]) / w.sum()
median_w = x[order][np.searchsorted(cum, 0.5)]     # weighted median
print(round(mean_w), median_w)

## B.7 NumPy → Ch 5, 25

In [ ]:
G = np.array([[100.0, 102.0], [100.0, 110.0]])     # rows: months; cols: groups
wts = np.array([0.6, 0.4])
print(G @ wts / wts.sum())                 # weighted index: matrix product
print(np.log(G).round(3), np.trunc(9.19 * 10) / 10)   # logs; truncation (Ch 9 §9.8)
rng = np.random.default_rng(17)            # reproducible random numbers (Ch 17)
print(rng.normal(size=3).round(3))

## B.8 A chart in the book's style → Ch 8

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(cpi["date"], cpi["inflation"], color="#1f4e79", lw=1.6, label="CPI inflation")
ax.step(repo["date"], repo["value"], where="post", color="#c55a11", lw=1.4,
        label="Repo rate")
ax.set_xlim(cpi["date"].min(), cpi["date"].max())
ax.set_ylim(0, 10)
ax.axhspan(2, 6, color="#9db7d5", alpha=0.25)
ax.set_title("Inflation and the policy rate, %", loc="left", fontsize=12)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.text(0.01, -0.03, "Source: MoSPI CPI (base 2012); BIS policy rates.",
         fontsize=8, color="grey")
plt.show()

## B.9 Regression with statsmodels → Ch 18–20

In [ ]:
import statsmodels.formula.api as smf

df = m.dropna(subset=["inflation", "repo"]).copy()
df["infl_lag"] = df["inflation"].shift(1)
df = df.dropna(subset=["infl_lag"])

ols = smf.ols("inflation ~ infl_lag + repo", data=df).fit()          # classical SEs
rob = smf.ols("inflation ~ infl_lag + repo", data=df).fit(cov_type="HC1")   # Stata's robust
hac = smf.ols("inflation ~ infl_lag + repo", data=df).fit(
    cov_type="HAC", cov_kwds={"maxlags": 3})                          # Newey–West (Ch 11)
pd.DataFrame({"coef": ols.params, "SE": ols.bse, "HC1": rob.bse,
              "HAC(3)": hac.bse}).round(3)

In [ ]:
df["above"] = (df["inflation"] > 6).astype(int)
logit = smf.logit("above ~ infl_lag", data=df).fit(disp=0)   # probit: smf.probit
ame = logit.get_margeff(at="overall").summary_frame()        # average marginal effects
ame.round(3)

## B.10 Panel data with linearmodels → Ch 21, 24

In [ ]:
from linearmodels.panel import PanelOLS

panel = pc.assign(ly=np.log(pc["lag"]), g=pc["growth"]).dropna(subset=["ly", "g"])
panel = panel.set_index(["state", "fy"])           # entity first, then time
fe = PanelOLS(panel["g"], panel[["ly"]], entity_effects=True).fit(
    cov_type="clustered", cluster_entity=True)
print(f"FE coefficient {fe.params['ly']:.2f} (clustered SE {fe.std_errors['ly']:.2f}), "
      f"{fe.nobs} obs")

## B.11 Time series → Ch 22, 23

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR

y = df.set_index("date")["inflation"].asfreq("MS")
print(f"ADF p-value: {adfuller(y.dropna(), autolag='AIC', result_object=False)[1]:.3f}")
ar = SARIMAX(y, order=(1, 0, 0), trend="c").fit(disp=False)
print(ar.forecast(3).round(2).to_string())

pre = df.set_index("date").loc["2014-01":"2020-03", ["inflation", "repo"]]
var = VAR(pre.asfreq("MS")).fit(2)       # no gaps allowed: stop before April 2020
print("Repo Granger-causes inflation, p =",
      round(var.test_causality("inflation", ["repo"]).pvalue, 3))

## B.12 SciPy: roots, optimisation, linear programmes → Ch 25, 26

In [ ]:
from scipy import optimize, stats

root = optimize.brentq(lambda p: 100 / p - (20 + 2 * p), 0.1, 50)  # demand = supply
best = optimize.minimize(lambda x: (x[0] - 2) ** 2 + (x[1] + 1) ** 2, x0=[0, 0])
lp = optimize.linprog(c=[2, 3], A_ub=[[-1, -1]], b_ub=[-10], bounds=[(0, None)] * 2,
                      method="highs")                   # min cost, x1 + x2 >= 10
print(round(root, 3), best.x.round(3), lp.x, round(stats.norm.ppf(0.975), 3))

## B.13 scikit-learn → Ch 29

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = pd.concat({f"lag{k}": y.shift(k) for k in range(1, 13)}, axis=1)
data = pd.concat([y.rename("y"), X], axis=1).dropna()
model = make_pipeline(StandardScaler(), LassoCV(cv=TimeSeriesSplit(5)))
model.fit(data.drop(columns="y"), data["y"])
kept = (model[-1].coef_ != 0).sum()
print(f"Lasso keeps {kept} of 12 lags")

## B.14 Habits